In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
DEVICE = torch.device("cpu")

In [16]:
import sys
import os
import yaml
from pathlib import Path

# Ruta al repo clonado
VITS2_DIR = Path(os.getcwd()) / "vits2"
sys.path.insert(0, str(VITS2_DIR))

# Cargar config
config_path = VITS2_DIR / "datasets" / "ljs_base" / "config.yaml"
with open(config_path, "r") as f:
    hps = yaml.safe_load(f)

model_cfg = hps["model"]
data_cfg  = hps["data"]

from model.models import SynthesizerTrn
from model.discriminator import MultiPeriodDiscriminator
from utils.task import load_vocab

# Vocabulario IPA (fonemas) — determina n_vocab correctamente
vocab = load_vocab(str(VITS2_DIR / "datasets" / "ljs_base" / "vocab.txt"))

# spec_channels: n_mels (80) cuando use_mel=True, de lo contrario n_fft//2+1 (1025)
spec_channels = data_cfg["n_mels"] if data_cfg.get("use_mel", True) else data_cfg["n_fft"] // 2 + 1

# ── Generador ─────────────────────────────────────────────────────────────────
net_g = SynthesizerTrn(
    n_vocab                  = len(vocab),
    spec_channels            = spec_channels,
    segment_size             = hps["train"]["segment_size"] // data_cfg["hop_length"],
    inter_channels           = model_cfg["inter_channels"],
    hidden_channels          = model_cfg["hidden_channels"],
    filter_channels          = model_cfg["filter_channels"],
    n_heads                  = model_cfg["n_heads"],
    n_layers                 = model_cfg["n_layers"],
    n_layers_q               = model_cfg["n_layers_q"],            # ← faltaba
    n_flows                  = model_cfg["n_flows"],               # ← faltaba
    kernel_size              = model_cfg["kernel_size"],
    p_dropout                = model_cfg["p_dropout"],
    speaker_cond_layer       = model_cfg["speaker_cond_layer"],    # ← faltaba
    resblock                 = model_cfg["resblock"],
    resblock_kernel_sizes    = model_cfg["resblock_kernel_sizes"],
    resblock_dilation_sizes  = model_cfg["resblock_dilation_sizes"],
    upsample_rates           = model_cfg["upsample_rates"],
    upsample_initial_channel = model_cfg["upsample_initial_channel"],
    upsample_kernel_sizes    = model_cfg["upsample_kernel_sizes"],
    mas_noise_scale          = model_cfg["mas_noise_scale"],       # ← faltaba
    mas_noise_scale_decay    = model_cfg["mas_noise_scale_decay"], # ← faltaba
    use_transformer_flow     = model_cfg.get("use_transformer_flow", False),
    n_speakers               = model_cfg.get("n_speakers", 0),
).to(DEVICE)

# ── Discriminador GAN ─────────────────────────────────────────────────────────
net_d = MultiPeriodDiscriminator(
    use_spectral_norm=model_cfg.get("use_spectral_norm", False)
).to(DEVICE)

total_g = sum(p.numel() for p in net_g.parameters()) / 1e6
total_d = sum(p.numel() for p in net_d.parameters()) / 1e6
print(f"✓ SynthesizerTrn:           {total_g:.1f}M params")
print(f"✓ MultiPeriodDiscriminator: {total_d:.1f}M params")
print(f"  vocab={len(vocab)} | spec_ch={spec_channels} | "
      f"segment={hps['train']['segment_size'] // data_cfg['hop_length']}")


CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:

In [ ]:
import subprocess
import shutil
import pandas as pd

# ── 1. Verificar espeak-ng (necesario para fonemización on-the-fly) ────────────
if not shutil.which("espeak-ng"):
    print("Instalando espeak-ng (puede tardar ~1 min)...")
    result = subprocess.run(["brew", "install", "espeak-ng"],
                            capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ espeak-ng instalado")
    else:
        print("✗ Error instalando espeak-ng. Instala manualmente: brew install espeak-ng")
        print(result.stderr[:300])
else:
    print(f"✓ espeak-ng disponible: {shutil.which('espeak-ng')}")

# ── 2. Crear filelists desde LJSpeech-1.1/metadata.csv ────────────────────────
#    El train.txt original (data/processed/texts/) tenía texto vacío (formato:
#    audio_path||speaker_id). Se reconstruye aquí usando los WAVs originales.
PROJECT_DIR  = Path(os.getcwd())
LJS_DIR      = PROJECT_DIR / "data" / "raw" / "LJSpeech-1.1"
FILELIST_DIR = PROJECT_DIR / "data" / "filelists"
FILELIST_DIR.mkdir(exist_ok=True)

# metadata.csv: id|raw_text|normalized_text (separador '|')
metadata = pd.read_csv(
    LJS_DIR / "metadata.csv", sep="|", header=None,
    names=["id", "text", "normalized_text"], quoting=3
)

# Split 90% train / 10% val
n_train  = int(len(metadata) * 0.9)
train_df = metadata.iloc[:n_train]
val_df   = metadata.iloc[n_train:]

def write_filelist(df, out_path):
    """Escribe filelist en formato: /ruta/audio.wav|texto_normalizado"""
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            wav_path = str(LJS_DIR / "wavs" / f"{row['id']}.wav")
            text     = str(row["normalized_text"]).strip()
            f.write(f"{wav_path}|{text}\n")

TRAIN_LIST = FILELIST_DIR / "ljspeech_train.txt"
VAL_LIST   = FILELIST_DIR / "ljspeech_val.txt"
write_filelist(train_df, TRAIN_LIST)
write_filelist(val_df,   VAL_LIST)

print(f"\n✓ Train filelist: {len(train_df):5d} muestras → {TRAIN_LIST}")
print(f"✓ Val   filelist: {len(val_df):5d} muestras → {VAL_LIST}")
print(f"\n  Ejemplo: {open(TRAIN_LIST).readline().strip()[:100]}")


In [ ]:
from types import SimpleNamespace
from data_utils import TextAudioLoader, TextAudioCollate
from torch.utils.data import DataLoader

# Config de datos apuntando a los filelists y WAVs originales de LJSpeech
data_hps = SimpleNamespace(
    training_files   = str(TRAIN_LIST),
    validation_files = str(VAL_LIST),
    vocab_file       = str(VITS2_DIR / "datasets" / "ljs_base" / "vocab.txt"),
    text_cleaners    = data_cfg["text_cleaners"],   # phonemize_text → espeak-ng
    sample_rate      = data_cfg["sample_rate"],
    n_fft            = data_cfg["n_fft"],
    hop_length       = data_cfg["hop_length"],
    win_length       = data_cfg["win_length"],
    n_mels           = data_cfg["n_mels"],
    f_min            = data_cfg["f_min"],
    f_max            = data_cfg.get("f_max"),
    use_mel          = data_cfg.get("use_mel", True),
    language         = data_cfg.get("language", "en-us"),
    cleaned_text     = False,   # texto crudo; fonemización on-the-fly con espeak-ng
    min_text_len     = 1,
    max_text_len     = 200,
)

BATCH_SIZE = 4   # Reducido para MPS; aumentar si la RAM lo permite

train_dataset = TextAudioLoader(data_hps.training_files, data_hps)
val_dataset   = TextAudioLoader(data_hps.validation_files, data_hps)
collate_fn    = TextAudioCollate()

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, drop_last=True,
    num_workers=0,  # 0 en MPS para evitar problemas con multiprocessing fork
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, drop_last=False, num_workers=0,
)

print(f"✓ Train: {len(train_dataset):5d} muestras | {len(train_loader):4d} batches/epoch")
print(f"✓ Val:   {len(val_dataset):5d} muestras | {len(val_loader):4d} batches")
print(f"  batch_size={BATCH_SIZE} | device={DEVICE}")


In [ ]:
import torch.nn.functional as F
from losses import (generator_loss, discriminator_loss,
                    feature_loss, kl_loss, kl_loss_normal)
from utils.mel_processing import wav_to_mel, spec_to_mel, spectral_norm as log_norm
from utils.model import slice_segments, clip_grad_value_

TRAIN_CFG = hps["train"]

optim_g = torch.optim.AdamW(
    net_g.parameters(),
    lr    = TRAIN_CFG["learning_rate"],
    betas = tuple(TRAIN_CFG["betas"]),
    eps   = TRAIN_CFG["eps"],
)
optim_d = torch.optim.AdamW(
    net_d.parameters(),
    lr    = TRAIN_CFG["learning_rate"],
    betas = tuple(TRAIN_CFG["betas"]),
    eps   = TRAIN_CFG["eps"],
)

scheduler_g = torch.optim.lr_scheduler.ExponentialLR(optim_g, gamma=TRAIN_CFG["lr_decay"])
scheduler_d = torch.optim.lr_scheduler.ExponentialLR(optim_d, gamma=TRAIN_CFG["lr_decay"])

CHECKPOINT_DIR = Path(os.getcwd()) / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

print("✓ Optimizadores AdamW")
print(f"  lr={TRAIN_CFG['learning_rate']} | betas={TRAIN_CFG['betas']} | "
      f"lr_decay={TRAIN_CFG['lr_decay']}")
print(f"✓ Checkpoints → {CHECKPOINT_DIR}")


In [ ]:
def train_epoch(epoch, net_g, net_d, loader, optim_g, optim_d,
                device, hps, max_steps=None):
    """Un epoch de entrenamiento GAN (generador + discriminador).

    Adaptado de vits2/train.py para un solo dispositivo (MPS/CPU/CUDA).
    Se elimina DDP y GradScaler (no soportado bien en MPS).

    Args:
        max_steps: limitar pasos por epoch (útil para pruebas). None = epoch completo.
    """
    net_g.train()
    net_d.train()
    train_cfg = hps["train"]
    data_cfg  = hps["data"]
    seg_size  = train_cfg["segment_size"] // data_cfg["hop_length"]

    totals = {"disc": 0.0, "gen": 0.0, "mel": 0.0, "fm": 0.0, "dur": 0.0}
    n_steps = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch:03d}", leave=False)
    for x, x_len, spec, spec_len, y, y_len in pbar:
        x,    x_len    = x.to(device),    x_len.to(device)
        spec, spec_len = spec.to(device), spec_len.to(device)
        y,    y_len    = y.to(device),    y_len.to(device)

        # ── Forward del generador ─────────────────────────────────────────────
        (y_hat, l_length, attn, ids_slice, x_mask, z_mask,
         (m_p_text, logs_p_text),
         (m_p_dur,  logs_p_dur, z_q_dur, logs_q_dur),
         (m_p_audio, logs_p_audio, m_q_audio, logs_q_audio),
        ) = net_g(x, x_len, spec, spec_len)

        # Mel del objetivo y del audio generado
        mel_target = log_norm(spec) if data_cfg.get("use_mel", True) else spec_to_mel(
            spec, data_cfg["n_fft"], data_cfg["n_mels"],
            data_cfg["sample_rate"], data_cfg["f_min"], data_cfg.get("f_max"))

        mel_gen = wav_to_mel(
            y_hat.squeeze(1),
            data_cfg["n_fft"],  data_cfg["n_mels"],  data_cfg["sample_rate"],
            data_cfg["hop_length"], data_cfg["win_length"],
            data_cfg["f_min"],  data_cfg.get("f_max"),
        )

        y_mel = slice_segments(mel_target, ids_slice, seg_size)
        y_wav = slice_segments(y, ids_slice * data_cfg["hop_length"],
                               train_cfg["segment_size"])

        # ── Paso del discriminador ────────────────────────────────────────────
        y_d_hat_r, y_d_hat_g, _, _ = net_d(y_wav, y_hat.detach())
        loss_disc, _, _ = discriminator_loss(y_d_hat_r, y_d_hat_g)

        optim_d.zero_grad()
        loss_disc.backward()
        clip_grad_value_(net_d.parameters(), None)
        optim_d.step()

        # ── Paso del generador ────────────────────────────────────────────────
        y_d_hat_r, y_d_hat_g, fmap_r, fmap_g = net_d(y_wav, y_hat)

        loss_dur      = torch.sum(l_length.float())
        loss_mel      = F.l1_loss(y_mel, mel_gen) * train_cfg["c_mel"]
        loss_gen, _   = generator_loss(y_d_hat_g)
        loss_kl_dur   = kl_loss(z_q_dur, logs_q_dur, m_p_dur, logs_p_dur,
                                z_mask) * train_cfg["c_kl_dur"]
        loss_kl_audio = kl_loss_normal(m_p_audio, logs_p_audio,
                                       m_q_audio,  logs_q_audio,
                                       z_mask) * train_cfg["c_kl_audio"]
        loss_fm       = feature_loss(fmap_r, fmap_g)
        loss_gen_all  = loss_gen + loss_fm + loss_mel + loss_dur + loss_kl_dur + loss_kl_audio

        optim_g.zero_grad()
        loss_gen_all.backward()
        clip_grad_value_(net_g.parameters(), None)
        optim_g.step()

        totals["disc"] += loss_disc.item()
        totals["gen"]  += loss_gen.item()
        totals["mel"]  += loss_mel.item()
        totals["fm"]   += loss_fm.item()
        totals["dur"]  += loss_dur.item()
        n_steps += 1
        pbar.set_postfix({k: f"{v/n_steps:.3f}" for k, v in totals.items()})

        if max_steps and n_steps >= max_steps:
            break

    return {k: v / n_steps for k, v in totals.items()}


# ── Configuración ─────────────────────────────────────────────────────────────
# NOTA: entrenamiento completo requiere miles de epochs (config: 20000).
#       Para una prueba funcional, usar N_EPOCHS=10, MAX_STEPS=50.
#       Para producción, poner MAX_STEPS=None y aumentar N_EPOCHS.
N_EPOCHS        = 10   # epochs a entrenar
MAX_STEPS_EPOCH = 50   # pasos por epoch (None = epoch completo)
SAVE_EVERY      = 5    # guardar checkpoint cada N epochs

history = []
for epoch in range(1, N_EPOCHS + 1):
    avg = train_epoch(epoch, net_g, net_d, train_loader,
                      optim_g, optim_d, DEVICE, hps,
                      max_steps=MAX_STEPS_EPOCH)
    scheduler_g.step()
    scheduler_d.step()
    history.append(avg)

    print(f"Epoch {epoch:03d} | "
          f"disc={avg['disc']:.3f}  gen={avg['gen']:.3f}  "
          f"mel={avg['mel']:.3f}  fm={avg['fm']:.3f}  dur={avg['dur']:.3f}")

    if epoch % SAVE_EVERY == 0:
        torch.save({"epoch": epoch,
                    "state_dict": net_g.state_dict(),
                    "optimizer":  optim_g.state_dict()},
                   CHECKPOINT_DIR / f"G_epoch{epoch:04d}.pth")
        torch.save({"epoch": epoch,
                    "state_dict": net_d.state_dict(),
                    "optimizer":  optim_d.state_dict()},
                   CHECKPOINT_DIR / f"D_epoch{epoch:04d}.pth")
        print(f"  → Checkpoint guardado en checkpoints/")

# ── Curva de pérdidas ─────────────────────────────────────────────────────────
if len(history) > 1:
    fig, axes = plt.subplots(1, len(history[0]), figsize=(15, 3))
    for ax, key in zip(axes, history[0].keys()):
        ax.plot([h[key] for h in history])
        ax.set_title(key)
        ax.set_xlabel("Epoch")
    plt.tight_layout()
    plt.show()

print("✓ Entrenamiento completado")


In [ ]:
import torchaudio
import IPython.display as ipd

def synthesize(text, model, vocab, data_cfg_dict, device,
               noise_scale=0.667, noise_scale_w=0.8, length_scale=1.0):
    """Genera audio a partir de texto usando el modelo entrenado.

    Args:
        noise_scale:   variación de la voz (0→determinista, 1→máxima variación)
        noise_scale_w: variación de la duración (ritmo)
        length_scale:  velocidad (>1 = más lento, <1 = más rápido)
    """
    from text import tokenizer as vits_tokenizer

    tokens = vits_tokenizer(
        text, vocab,
        cleaner_names = data_cfg_dict["text_cleaners"],
        language      = data_cfg_dict.get("language", "en-us"),
        cleaned_text  = False,
    )
    x        = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    x_lengths = torch.tensor([len(tokens)], dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        wav, attn, *_ = model.infer(
            x, x_lengths,
            noise_scale   = noise_scale,
            noise_scale_w = noise_scale_w,
            length_scale  = length_scale,
        )
    return wav.squeeze().cpu().float()


# ── Síntesis de ejemplo ───────────────────────────────────────────────────────
test_texts = [
    "Printing, in the only sense with which we are at present concerned.",
    "Hello! This is a voice synthesis demonstration.",
]

sr = data_cfg["sample_rate"]
for i, text in enumerate(test_texts):
    print(f"Sintetizando: \"{text}\"")
    wav = synthesize(text, net_g, vocab, data_cfg, DEVICE)
    out_path = f"sample_{i + 1}.wav"
    torchaudio.save(out_path, wav.unsqueeze(0), sr)
    print(f"  ✓ Guardado: {out_path}")
    ipd.display(ipd.Audio(wav.numpy(), rate=sr))
